In [ ]:
import cv2
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow

# Choose an image from your PC.
uploaded = files.upload()

if not uploaded:
    print("No image selected. Run this cell again.")
else:
    filename = next(iter(uploaded))

    # Convert the uploaded file into an image OpenCV can read.
    image_bytes = np.frombuffer(uploaded[filename], dtype=np.uint8)
    image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

    if image is None:
        print("Could not read this file. Try a JPG or PNG.")
    else:
        height, width = image.shape[:2]

        print(f"File: {filename}")
        print(f"Dimensions: {width} × {height} pixels")

        # Make a smaller preview without changing the original.
        scale = min(1.0, 800 / max(height, width))
        preview = cv2.resize(
            image,
            (max(1, round(width * scale)),
             max(1, round(height * scale)))
        )

        cv2_imshow(preview)

In [ ]:
%pip install -q transformers pillow

In [ ]:
from transformers import pipeline

# Keep the original detector so we can compare both.
detector_v2 = pipeline(
    "image-classification",
    model="Ateeqq/ai-vs-human-image-detector",
    device=-1
)

print("Second detector loaded!")

In [ ]:
from PIL import Image, ImageOps
import io

# Read your latest uploaded image.
filename = next(iter(uploaded))
photo = Image.open(io.BytesIO(uploaded[filename]))
photo = ImageOps.exif_transpose(photo).convert("RGB")

display(photo)

# Compare both models on exactly the same image.
for name, model in [
    ("Second detector", detector_v2)
]:
    print(f"\n{name}:")
    for result in model(photo, top_k=2):
        print(f"  {result['label']}: {result['score']:.6f}")

In [ ]:
%pip install -q gradio

In [ ]:
import gradio as gr
from PIL import ImageOps

def analyze_image(image):
    if image is None:
        raise gr.Error("Please upload an image first.")

    # Prepare the uploaded image.
    image = ImageOps.exif_transpose(image).convert("RGB")

    # Run the Ateeqq model.
    results = detector_v2(image, top_k=2)

    names = {
        "ai": "AI-generated",
        "hum": "Human / camera"
    }

    return {
        names.get(item["label"], item["label"]): item["score"]
        for item in results
    }

app = gr.Interface(
    fn=analyze_image,
    inputs=gr.Image(
        type="pil",
        sources=["upload"],
        label="Upload an image"
    ),
    outputs=gr.Label(
        num_top_classes=2,
        label="Model scores — not verified probabilities"
    ),
    title="AI Image Detector",
    description=(
        "Upload an image to explore whether the model classifies "
        "it as AI-generated or human/camera. "
        "This experimental detector can misclassify real photos. "
        "It does not check for Photoshop edits or copied regions."
    ),
    article=(
        "Model: Ateeqq/ai-vs-human-image-detector on Hugging Face."
    )
)

app.launch(share=True)